# Variante Kaggle — J3

Importer ce notebook dans Kaggle. Ajouter les données officielles **playground-series-s6e11** via Add Input après acceptation personnelle des règles. Choisir CPU pour reproduire le budget local. Lancer les cellules dans l’ordre, puis conserver les sorties de `/kaggle/working/MyLittleMLLab/runs/J3`. Les numéros de cellules du PowerPoint concernent la version locale ; cette variante ajoute deux cellules d’amorçage.

Aucun secret ni token n’est inclus. La métrique et les folds restent identiques. Un dépôt éventuel se fait manuellement. Vérifier les règles, les limites du compte et l’ouverture du concours avant la séance.


In [ ]:
# Adaptateur Kaggle : mêmes cellules métier, mêmes données et mêmes folds.
from pathlib import Path
import os,base64,io,zipfile,shutil
out=Path(os.environ.get('MLLAB_KAGGLE_OUTPUT','/kaggle/working'))/'MyLittleMLLab'
out.mkdir(parents=True,exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(out)
inputs=Path(os.environ.get('MLLAB_KAGGLE_INPUT','/kaggle/input'))
slug='playground-series-s6e11'
matches=[p.parent for p in inputs.rglob('sample_submission.csv') if slug in str(p.parent) and (p.parent/'train.csv').exists() and (p.parent/'test.csv').exists()]
if len(matches)!=1:raise FileNotFoundError('Ajouter les données officielles playground-series-s6e11 via Add Input. Un seul jeu correspondant doit être présent.')
dest=out/'data/competitions'/slug;dest.mkdir(parents=True,exist_ok=True)
for name in ['train.csv','test.csv','sample_submission.csv']:
 shutil.copy2(matches[0]/name,dest/name)
os.chdir(out)
print('Laboratoire préparé dans le répertoire de travail Kaggle ; aucun téléchargement ou dépôt automatique.')


# AML_03 — Arbres, ensembles et budget de recherche

**QUID :** quelle complexité améliore le classement des fraudes pour un coût acceptable ?

Même compétition, mêmes données officielles et mêmes folds que J2. Les scores sont locaux ; aucune soumission automatique.

Chaque numéro « cellule » désigne la position physique dans ce notebook, en comptant les textes. Exécuter dans l’ordre. Installation conservée depuis le ZIP initial ; aucun git pull.

Aucune nouvelle bibliothèque.

[Guide de démarrage](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/DEMARRER_J3.md) · [Cours rédigé](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/lectures/AML_03_cours.md)


## Le contrat de comparaison

On réutilise le split et les folds. La validation externe déjà vue est un contrôle de développement.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
from pathlib import Path
import sys, json, time, hashlib, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import subsets, make_model, xy, SPECS
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
LIMIT=int(os.environ.get('MLLAB_LIMIT','15000'))
tr,va,test,sample,protocol_record=subsets('fraud',root=ROOT,limit=LIMIT)
X,y=xy(tr,'fraud'),tr.is_fraud.astype(int)
Xv,yv=xy(va,'fraud'),va.is_fraud.astype(int)
folds=list(StratifiedKFold(3,shuffle=True,random_state=2026).split(X,y))
assert not set(tr.id)&set(va.id)
RUN=ROOT/'runs/J3';RUN.mkdir(parents=True,exist_ok=True)
ids_sha=hashlib.sha256(tr.id.to_numpy().tobytes()).hexdigest()
print('Train :',len(tr),'fraudes :',int(y.sum()),'| contrôle local :',len(va))
print('3 folds fixes ; ROC-AUC ; CPU ; même protocole que J2.')
print('Le contrôle local a déjà été consulté en J2 : il ne constitue plus un test vierge.')


## Une règle et une feuille

Exemple miniature distinct des données Kaggle. Le seuil 35 est imposé pour calculer ; le logiciel choisit librement son seuil.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
montants=np.array([10,20,30,40,50,60]).reshape(-1,1)
classes=np.array([0,0,1,0,1,1])
gauche=classes[montants[:,0]<=35];droite=classes[montants[:,0]>35]
print('Règle imposée 35 : gauche =',gauche.mean(),'droite =',droite.mean())
arbre=DecisionTreeClassifier(max_depth=1,random_state=2026).fit(montants,classes)
print('Seuil appris :',arbre.tree_.threshold[0])
plt.figure(figsize=(10,4));plot_tree(arbre,feature_names=['montant'],class_names=['normal','fraude'],filled=True);plt.show()

## La profondeur et la mémorisation

Les scores train servent au diagnostic ; le choix repose sur la CV. Dans cet exemple jouet, un arbre profond peut mémoriser.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
lignes=[]
for depth in [1,2,None]:
 m=DecisionTreeClassifier(max_depth=depth,random_state=2026).fit(montants,classes)
 lignes.append({'profondeur':str(depth),'feuilles':m.get_n_leaves(),'accuracy_train_jouet':m.score(montants,classes)})
display(pd.DataFrame(lignes))
print('Trois arbres donnent .2, .5, .8 : moyenne des probabilités =',np.mean([.2,.5,.8]))

## Les candidats et les catégories

Les codes de catégories peuvent être des étiquettes ou des nombres ordonnés. Nous comparons les deux interprétations avec le même algorithme.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
num=X.select_dtypes(include='number').columns.tolist()
cat=[c for c in X if c not in num]
def native_boost(leaves=15,rate=.1):
 prep=ColumnTransformer([
  ('num',SimpleImputer(strategy='median',keep_empty_features=True),num),
  ('cat',make_pipeline(SimpleImputer(strategy='most_frequent',keep_empty_features=True),OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)),cat)])
 # Valeurs négatives inconnues traitées comme manquantes par HistGradientBoosting.
 mask=[False]*len(num)+[True]*len(cat)
 return make_pipeline(prep,HistGradientBoostingClassifier(categorical_features=mask,max_iter=80,max_leaf_nodes=leaves,learning_rate=rate,early_stopping=False,random_state=2026))
candidates={
 'logistique_C01':make_model('fraud',tr,'linear',params={'C':.1}),
 'foret_40':make_model('fraud',tr,'forest',params={'n_estimators':40,'max_depth':8}),
 'boost_ordinal':make_model('fraud',tr,'boosting',params={'early_stopping':False}),
 'boost_categoriel':native_boost()}
print('4 candidats × 3 folds = 12 fits. Catégories :',len(cat))
print('La comparaison encodage ordinal / catégories natives utilise les mêmes 80 étapes et 15 feuilles.')

## Le tournoi sur trois folds

On calcule moyenne, dispersion et durée. Garder tous les résultats, y compris les essais moins bons.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
rows=[];fold_rows=[]
for name,model in candidates.items():
 start=time.perf_counter()
 result=cross_validate(model,X,y,cv=folds,scoring='roc_auc',return_train_score=True,n_jobs=1)
 rows.append({'modele':name,'auc_cv':result['test_score'].mean(),'ecart_type':result['test_score'].std(),'auc_train':result['train_score'].mean(),'secondes':time.perf_counter()-start})
 for k,s in enumerate(result['test_score'],1):fold_rows.append({'modele':name,'fold':k,'auc':float(s)})
tournoi=pd.DataFrame(rows).sort_values('auc_cv',ascending=False)
display(tournoi.round(5));tournoi.to_csv(RUN/'tournoi.csv',index=False)
pd.DataFrame(fold_rows).to_csv(RUN/'folds.csv',index=False)

## Une recherche bornée

Trois configurations natives, trois folds : neuf fits, sans refit automatique. On accepte qu’aucune amélioration ne soit démontrée.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
configs=[{'leaves':7,'rate':.1},{'leaves':15,'rate':.1},{'leaves':15,'rate':.03}]
search_rows=[]
for cfg in configs:
 start=time.perf_counter();res=cross_validate(native_boost(**cfg),X,y,cv=folds,scoring='roc_auc',n_jobs=1)
 search_rows.append({**cfg,'auc_cv':res['test_score'].mean(),'ecart_type':res['test_score'].std(),'secondes':time.perf_counter()-start})
recherche=pd.DataFrame(search_rows);display(recherche.round(5))
recherche.to_csv(RUN/'recherche.csv',index=False)
print('Budget réalisé : 12 + 9 = 21 fits CV. Les neuf fits incluent une répétition de la configuration de référence.')

## La décision avant le contrôle

Le choix utilise la moyenne CV. Un faible écart ne prouve pas une supériorité générale. Aucun nouveau réglage à partir du contrôle.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
best_cfg=recherche.sort_values('auc_cv',ascending=False).iloc[0]
best_base=tournoi.iloc[0]
if best_cfg.auc_cv>best_base.auc_cv:
 selected_name='boost_categoriel_regle';selected_model=native_boost(int(best_cfg.leaves),float(best_cfg.rate))
else:
 selected_name=str(best_base.modele);selected_model=clone(candidates[selected_name])
print('Candidat retenu par la CV :',selected_name)
selected_model.fit(X,y)
controle=roc_auc_score(yv,selected_model.predict_proba(Xv)[:,1])
print('AUC du contrôle local déjà utilisé en J2 :',round(controle,6))
print('Ce résultat est descriptif ; pas une estimation indépendante finale.')

## Interpréter une variable

On permute une variable sur les lignes de contrôle, puis on mesure la baisse d’AUC. Une importance ne prouve pas une causalité.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
from sklearn.inspection import permutation_importance
imp=permutation_importance(selected_model,Xv,yv,scoring='roc_auc',n_repeats=2,random_state=2026,n_jobs=1)
importance=pd.DataFrame({'variable':X.columns,'baisse_auc':imp.importances_mean,'dispersion':imp.importances_std}).sort_values('baisse_auc',ascending=False)
display(importance.head(8).round(5));importance.to_csv(RUN/'importance.csv',index=False)
print('Variables corrélées : une permutation isolée peut sous-estimer leur utilité commune.')

## Le CSV Kaggle et sa preuve

Export réel sur tout le test officiel, sans dépôt automatique. Le modèle est refit sur un échantillon stratifié du train officiel, après le choix.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
from sklearn.model_selection import train_test_split
from mllab.competition import read_competition
all_train,test,sample=read_competition('fraud',root=ROOT)
fit_train,_=train_test_split(all_train,train_size=LIMIT,stratify=all_train.is_fraud,random_state=2026)
final_model=clone(selected_model).fit(xy(fit_train,'fraud'),fit_train.is_fraud)
pred=final_model.predict_proba(xy(test,'fraud'))[:,1]
submission=sample.copy();submission['is_fraud']=pred
assert list(submission)==['id','is_fraud'] and np.array_equal(submission.id,test.id)
assert np.isfinite(pred).all() and ((pred>=0)&(pred<=1)).all()
submission.to_csv(RUN/'submission_J3.csv',index=False)
manifest={'modele':selected_name,'parametres':str(selected_model.get_params()),'train_ids_sha256':ids_sha,'donnees_sha256':protocol_record['data_sha256'],'fits_cv':21,'controle_auc':controle,'lignes_submission':len(submission),'submission_sha256':hashlib.sha256((RUN/'submission_J3.csv').read_bytes()).hexdigest(),'score_public':None}
(RUN/'manifest.json').write_text(json.dumps(manifest,indent=2))
display(submission.head());print(len(submission),'lignes exportées ; aucun envoi à Kaggle.')

## Le journal et la prochaine expérience

Compléter les trois champs avant une nouvelle recherche ; le score public reste inconnu tant qu’aucun retour réel n’a été obtenu.

Avant calcul : prévoir une sortie. Après calcul : nommer le résultat, sa limite et la décision.

In [ ]:
journal={'hypothese':'À compléter avant une nouvelle expérience','preuve':'À compléter avec deux scores et un temps','decision':'À compléter : conserver ou abandonner, et pourquoi','prochaine_experience':'Un seul changement et un budget annoncé','score_public_reel':None}
(RUN/'journal.json').write_text(json.dumps(journal,indent=2,ensure_ascii=False))
print('Fichiers :',', '.join(sorted(p.name for p in RUN.iterdir() if p.is_file())))

## Remise et explication

Conserver le tableau des résultats, le manifeste, le CSV de prédictions et le journal. Compléter le journal avec une hypothèse, une conclusion chiffrée et une prochaine expérience. Chaque membre explique un choix et une limite. Vérifier les règles et l’ouverture de la compétition avant un dépôt manuel ; le CSV reste utilisable pour un exercice de format si le dépôt est fermé.
